# Claude Vision Example

**HFA AI Training — Module 6: Vision Language Models**

Claude has excellent vision capabilities, especially for:
- Document and form parsing (receipts, contracts, invoices)
- Chart and graph interpretation
- Detailed, careful image analysis
- Structured data extraction

This notebook demonstrates:
1. Sending an image to Claude via base64 encoding
2. Sending an image to Claude via URL
3. Analyzing a photo and getting a description
4. Extracting structured data from a document image

**Get a key at:** https://console.anthropic.com/

## Install Dependencies

In [ ]:
!pip install anthropic httpx

## Imports and Configuration

Load the required libraries and configure the Anthropic API key from environment variables.
Never hard-code API keys — always use environment variables.

In [ ]:
import os
import json
import base64
import httpx
from pathlib import Path

from anthropic import Anthropic

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

# Always load API keys from environment variables — never hard-code them.
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY not set. Run: export ANTHROPIC_API_KEY='your-key'\n"
        "Get a key at https://console.anthropic.com/"
    )

# Create the Anthropic client.
client = Anthropic(api_key=ANTHROPIC_API_KEY)

# Use Claude's latest model with vision support.
MODEL_NAME = "claude-sonnet-4-20250514"

## Helper: Load Image as Base64

Claude requires images to be sent as base64 or via URL.
This helper handles the encoding for local files.

In [ ]:
def image_to_base64(image_path: str) -> tuple[str, str]:
    """
    Read a local image file and return it as a base64-encoded string.

    Claude requires images to be sent as base64 or via URL.
    This helper handles the encoding.

    Args:
        image_path: Path to the image file.

    Returns:
        Tuple of (base64_string, media_type).
    """
    path = Path(image_path)

    # Determine the media type from the file extension.
    extension_to_media_type = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    media_type = extension_to_media_type.get(
        path.suffix.lower(), "image/jpeg"  # Default to JPEG
    )

    # Read the file and encode to base64.
    image_data = path.read_bytes()
    base64_string = base64.standard_b64encode(image_data).decode("utf-8")

    return base64_string, media_type

## Example 1: Analyze an Image from a URL

Claude can accept images directly via URL — no need to download
and encode them yourself. This is the simplest approach for web images.

In [ ]:
def analyze_image_from_url(image_url: str, question: str) -> str:
    """
    Send an image URL to Claude for analysis.

    Claude can accept images directly via URL — no need to download
    and encode them yourself. This is the simplest approach for web images.

    Args:
        image_url: Public URL of the image.
        question: What you want to know about the image.

    Returns:
        Claude's analysis text.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 1: Analyze an image from URL")
    print(f"{'='*60}")
    print(f"Image URL: {image_url}")
    print(f"Question:  {question}")

    # Claude's Messages API accepts images as content blocks.
    # For URLs, use the "url" source type.
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": [
                    # Image block — Claude will "see" this image.
                    {
                        "type": "image",
                        "source": {
                            "type": "url",
                            "url": image_url,
                        },
                    },
                    # Text block — your question about the image.
                    {
                        "type": "text",
                        "text": question,
                    },
                ],
            }
        ],
    )

    result = response.content[0].text
    print(f"\nClaude's Analysis:\n{result}")
    return result

## Example 2: Analyze a Local Image File (Base64)

Load a local image, encode it as base64, and send it to Claude.
This is how you analyze images stored on your computer or server.

In [ ]:
def analyze_local_image(image_path: str, question: str) -> str:
    """
    Load a local image, encode it as base64, and send it to Claude.

    This is how you analyze images stored on your computer or server.
    The image is sent inline as base64-encoded data.

    Args:
        image_path: Path to a local image file.
        question: What you want to know about the image.

    Returns:
        Claude's analysis text.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 2: Analyze a local image file (base64)")
    print(f"{'='*60}")
    print(f"Image path: {image_path}")
    print(f"Question:   {question}")

    # Convert the local file to base64.
    base64_data, media_type = image_to_base64(image_path)

    # Send to Claude with base64 source type.
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": [
                    # Image block with base64 data.
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": base64_data,
                        },
                    },
                    # Your question.
                    {
                        "type": "text",
                        "text": question,
                    },
                ],
            }
        ],
    )

    result = response.content[0].text
    print(f"\nClaude's Analysis:\n{result}")
    return result

## Example 3: Extract Structured Data from a Document Image

Claude excels at document parsing — this is "OCR on steroids."
Instead of just reading text, Claude UNDERSTANDS the document structure
and can organize the information into clean JSON.

Great for:
- Receipts -> line items, totals, tax
- Business cards -> name, phone, email, company
- Invoices -> vendor, items, amounts, due date
- Property listings -> features, price, dimensions

In [ ]:
def extract_structured_data(image_url: str) -> dict:
    """
    Ask Claude to extract structured JSON data from a document image.

    Claude excels at document parsing — this is "OCR on steroids."
    Instead of just reading text, Claude UNDERSTANDS the document structure
    and can organize the information into clean JSON.

    Great for:
      - Receipts -> line items, totals, tax
      - Business cards -> name, phone, email, company
      - Invoices -> vendor, items, amounts, due date
      - Property listings -> features, price, dimensions

    Args:
        image_url: URL of a document image.

    Returns:
        Parsed dictionary of extracted data.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 3: Structured data extraction from a document")
    print(f"{'='*60}")

    # The prompt is key — be specific about the JSON structure you want.
    extraction_prompt = """Analyze this image carefully and extract all
    information into a structured JSON format.

    If this is a property or real estate image:
    {
        "property_type": "house/condo/apartment/land",
        "style": "architectural style description",
        "condition_assessment": "detailed condition notes",
        "features": ["list", "of", "visible", "features"],
        "materials": ["visible construction materials"],
        "potential_issues": ["any concerns visible"],
        "listing_highlights": ["top selling points for a listing"],
        "estimated_era": "approximate decade or period of construction"
    }

    If this is a document (receipt, form, invoice, etc.):
    {
        "document_type": "type of document",
        "key_fields": { ... extracted fields ... },
        "line_items": [ ... if applicable ... ],
        "totals": { ... if applicable ... }
    }

    Return ONLY valid JSON — no markdown, no extra text."""

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2048,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "url",
                            "url": image_url,
                        },
                    },
                    {
                        "type": "text",
                        "text": extraction_prompt,
                    },
                ],
            }
        ],
    )

    raw_text = response.content[0].text.strip()

    # Claude sometimes wraps JSON in markdown code blocks.
    if raw_text.startswith("```"):
        raw_text = raw_text.split("\n", 1)[1]
        raw_text = raw_text.rsplit("```", 1)[0]

    try:
        structured_data = json.loads(raw_text)
        print(f"\nExtracted structured data:")
        print(json.dumps(structured_data, indent=2))
        return structured_data
    except json.JSONDecodeError:
        print(f"\nClaude returned text (not strict JSON):\n{raw_text}")
        return {"raw_response": raw_text}

## Example 4: Multi-Image Comparison

Send two images to Claude and ask it to compare them.

Useful for:
- Before/after comparisons
- Comparing two properties side by side
- Spotting differences between document versions

In [ ]:
def compare_two_images(image_url_1: str, image_url_2: str) -> str:
    """
    Send two images to Claude and ask it to compare them.

    Useful for:
      - Before/after comparisons
      - Comparing two properties side by side
      - Spotting differences between document versions

    Args:
        image_url_1: URL of the first image.
        image_url_2: URL of the second image.

    Returns:
        Claude's comparison analysis.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 4: Compare two images")
    print(f"{'='*60}")

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2048,
        messages=[
            {
                "role": "user",
                "content": [
                    # First image
                    {
                        "type": "image",
                        "source": {"type": "url", "url": image_url_1},
                    },
                    # Second image
                    {
                        "type": "image",
                        "source": {"type": "url", "url": image_url_2},
                    },
                    # Comparison question
                    {
                        "type": "text",
                        "text": (
                            "Compare these two images in detail. "
                            "What are the similarities and differences? "
                            "If these are properties, which would you "
                            "rate higher and why?"
                        ),
                    },
                ],
            }
        ],
    )

    result = response.content[0].text
    print(f"\nClaude's Comparison:\n{result}")
    return result

## Run All Examples

Execute all the Claude vision examples using sample public domain house photos.

In [ ]:
print("=" * 60)
print("  CLAUDE VISION EXAMPLES")
print("  Anthropic Claude — Excellent Document & Chart Analysis")
print("=" * 60)

# A sample image URL — a public domain house photo.
sample_image_url = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "1/19/Weatherboard_house_in_Coorparoo%2C_Queensland.jpg/"
    "1280px-Weatherboard_house_in_Coorparoo%2C_Queensland.jpg"
)

### Example 1: Analyze image from URL

In [ ]:
analyze_image_from_url(
    image_url=sample_image_url,
    question="Describe this property in detail, as if you were writing a real estate listing.",
)

### Example 2: Analyze local image

Uncomment and provide a real file path to test.

In [ ]:
# analyze_local_image(
#     image_path="/path/to/your/image.jpg",
#     question="What do you see in this image?",
# )

### Example 3: Structured data extraction

In [ ]:
extract_structured_data(sample_image_url)

### Example 4: Compare two images

In [ ]:
second_image_url = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "e/e1/Ascot_house_1.jpg/"
    "1280px-Ascot_house_1.jpg"
)
compare_two_images(sample_image_url, second_image_url)

In [ ]:
print("\n" + "=" * 60)
print("  All examples complete!")
print("=" * 60)